# Local Grocery – 100 customer selection for spacial offer

**Goal:**  Identify 100 customers with the highest likelihood to respond to a limited-time promotion.

**Deliverables:**
- `top_100_customers.csv` (ranked list with scores)
- This notebook as documentation (can be reused across countries/tenants)

## Data Overview

**Files Provided**
| Files Provided | Data  | Attributes |
|:------------- |:---------------| :-------------|
| businesscase_account.csv  | Account info  | 'mandant_id', 'Kundennummer', 'crm_konto_typ' |
| businesscase_customer.csv  | Customer profiles          | 'mandant_id', 'geburtsjahr', 'haushaltsgroesse', 'vorzugsfiliale_wup_id', 'zeit_registrierung', 'Kundennummer'  |
| businesscase_articles.csv  | Products  | 'mandant_id', 'kl_art_id', 'abt_bez', 'art_id', 'hwg_bez'|
| businesscase_promotion_customer.csv  | Promo history  | 'mandant_id', 'kl_art_id', 'bon_zeile', 'betrag_promo_brutto', 'umsatz_promo_brutto', 'menge_promo', 'Kundennummer', 'Transaktionsnummer'  |
| businesscase_sales_customer.csv  | Sales history  | 'mandant_id', 'umsatz_brutto', 'menge', 'kl_art_id', 'bon_zeile', 'einh_id', 'erfassung_id', 'bonus_fg', 'aktion_fg', 'wup_id', 'k_bon_dat', 'k_bon_beginn', 'k_bon_ende', 'k_kasse_typ_id', 'Kundennummer', 'Transaktionsnummer'|

**Key assumption:** Customer ID is the join key across all fact tables. Dates are in `k_bon_dat`.

## Approach Summary (RFM + Promo Engagement)

### Factors considered:
- Wide variety of customers with different activity levels
- Task is to find the most loyal and high-spending customers
- Campaign should prioritize exactly 100 customers

We compute:
- **Recency**: days since last purchase
- **Frequency**: # transactions in the last 90 days
- **Monetary**: total spend in the last 90 days
- **Promo Ratio**: share of transactions that were promo‑tagged (last 90 days)
- **Promo Value Ratio**: share of spend on promo items (last 90 days)

Then assign quintile scores (1–5) per metric and sum to a **Total Score**. Select the Top‑100 by Total Score.

## Environment & Imports

Import core libraries for data loading, prep, and transformations.

In [ ]:
import pandas as pd
import numpy as np
import re


/var/folders/4r/bw219wp92vz9b4fqkv_twq4m0000gn/T/ipykernel_13784/3420871739.py:9: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  promo = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/businesscase_promotion_customer.csv', sep = ';')


## Load Raw Data

Load all five CSVs using the provided separators. Paths are placeholders — adjust for your environment.

In [ ]:
### 1. Load all datasets ###
account = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/businesscase_account.csv', sep=';')
customer = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/businesscase_customer.csv', sep = ';')
sales = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/businesscase_sales_customer.csv', sep = ';')
articles = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/businesscase_articles.csv', sep = ';')
promo = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/businesscase_promotion_customer.csv', sep = ';')   

In [ ]:
account.columns
#print(account)

     mandant_id  Kundennummer crm_konto_typ
0             1           729       digital
1             1            38       digital
2             1           951       digital
3             1           821       digital
4             1           194       digital
..          ...           ...           ...
995           1           449       digital
996           1           771      physisch
997           1           534       digital
998           1           273       digital
999           1            48       digital

[1000 rows x 3 columns]
     client_id  customer_id crm_account_type
0            1          729          digital
1            1           38          digital
2            1          951          digital
3            1          821          digital
4            1          194          digital
..         ...          ...              ...
995          1          449          digital
996          1          771         physisch
997          1          534          digit

In [ ]:
customer.columns
#print(customer.head())

In [169]:
sales.columns
#print(sales.head())


        client_id sales_gross  quantity  product_cluster_id  \
0               1      49.900       1.0             1487236   
1               1      14.900       1.0             2903306   
2               1      0.6900       1.0             2525574   
3               1      21.800       2.0             1380588   
4               1      13.900       1.0             1018569   
...           ...         ...       ...                 ...   
728538          1      0.8900       1.0             2401964   
728539          1      0.8900       1.0             1401584   
728540          1      11.900       1.0             1378018   
728541          1      0.9400       1.0             1388491   
728542          1      19.900       1.0             1156848   

        receipt_line_number unit_id recording_id  bonus_flag  promotion_flag  \
0                        28      ST           SC           0               1   
1                        29      ST           SC           0               0   
2  

In [167]:
promo.columns 
#promo


,client_id,product_cluster_id,receipt_line_number,promo_amount_gross,promo_sales_gross,promo_quantity,customer_id,transaction_id
0,1,991599,21,-0.4100,-45.800,-1.0,411,8294
1,1,1346790,11,-0.1700,-23.400,-1.0,411,3041
2,1,918122,12,-0.0400,-13.500,-1.0,910,2616
3,1,1467208,12,-0.2000,-0.9900,-1.0,794,728
4,1,1273948,17,-0.0100,-0.2900,-1.0,794,2140
...,...,...,...,...,...,...,...,...
83344,1,13143,25,-0.08,-12.700,-1.0,319,5467
83345,1,1236129,12,-0.25,-30.000,-1.0,717,13204
83346,1,1229338,29,-0.03,-12.600,-1.0,118,28216
83347,1,1111021,33,-0.31,-36.800,-1.0,209,12195


In [168]:
articles.columns 
#articles


,client_id,product_cluster_id,category_name,product_id,product_group
0,1,1203135,Non Food,120313500,"Freizeit, Lernen, Mobilitaet"
1,1,1337102,Food Frische,133710200,Tiefkuehlkost
2,1,1503512,Food Frische,150351200,Tiefkuehlkost
3,1,2057051,Food,205705100,Alkoholfreie Getraenke
4,1,1532730,Food Frische,153273000,Tiefkuehlkost
...,...,...,...,...,...
40236,1,13769,Ultra-Frische,1376900,Fleisch / Fisch SB
40237,1,1283552,Ultra-Frische,128355200,Obst / Gemuese / Pflanzen
40238,1,1121798,Food Frische,112179800,Tiefkuehlkost
40239,1,1502522,Food,150252200,Beauty/Healthcare/Baby


##  Verify the dataset with glossary (Column Standardization German → English)

Use `glossary.csv` to translate/standardize column names. This ensures consistent schema across markets and facilitates reuse.

Rename columns in **all** datasets using the glossary mapping to English field names for clarity and downstream joins.

In [ ]:
### 2. Load glossary for column name interpretation ###
glossary = pd.read_csv('/Users/july/Desktop/Bewerbung/Kaufland/Businesscase_Kaufland/data/glossary.csv', sep=';')
rename_dict = glossary.set_index("original_column_name")["english_column_name"].to_dict()

### 3. Standardize column names across all datasets ###
account = account.rename(columns=rename_dict)
customer = customer.rename(columns=rename_dict)
sales = sales.rename(columns=rename_dict)
promo = promo.rename(columns=rename_dict)
articles = articles.rename(columns=rename_dict)
print(account.head())
print(customer.head())
print(sales.head())
print(promo.head())
print(articles.head())

## Data Quality Check: Product Master Coverage

Validate that every `product_cluster_id` present in **sales** exists in **articles** (product master).  
We log the count of missing references for transparency.

In [ ]:
### 4. Check for mismatches in product IDs between sales and article reference ###
missing_articles_in_sales = sales[~sales['product_cluster_id'].isin(articles['product_cluster_id'].unique())]

print(f"Number of sales rows with missing product IDs: {len(missing_articles_in_sales)}")

Количество строк с несуществующими товарами: 0


In [177]:
print(len(customer))
print(len(promo['customer_id'].unique()))

1000
897


In [178]:
print(promo.columns)
print(customer.columns)

Index(['client_id', 'product_cluster_id', 'receipt_line_number',
       'promo_amount_gross', 'promo_sales_gross', 'promo_quantity',
       'customer_id', 'transaction_id'],
      dtype='object')
Index(['client_id', 'birth_year', 'household_size', 'preferred_store_id',
       'registration_time', 'customer_id'],
      dtype='object')


## Data Quality Check: Orphan Promo Customers

Find customers appearing in **promo** but missing in **customer** master.  
These orphans can indicate incomplete imports; we keep logging but proceed with available customers.

In [ ]:
### 5. Identify promo customers not present in main customer table ###
matched_customers_promo = promo[~promo['customer_id'].isin(customer['customer_id'])]
matched_customers_promo

,client_id,product_cluster_id,receipt_line_number,promo_amount_gross,promo_sales_gross,promo_quantity,customer_id,transaction_id


## Normalize Product Cluster IDs

Trim whitespace and enforce string type for product cluster IDs in both **sales** and **articles** to avoid false mismatches.

In [ ]:
# Check the product error
# articles[articles['product_cluster_id'] == '1299925']

,client_id,product_cluster_id,category_name,product_id,product_group


In [ ]:
# Check the product error
# sales[sales['sales_gross'] == '1.290.000']

,mandant_id,umsatz_brutto,menge,kl_art_id,bon_zeile,einh_id,erfassung_id,bonus_fg,aktion_fg,wup_id,k_bon_dat,k_bon_beginn,k_bon_ende,k_kasse_typ_id,Kundennummer,Transaktionsnummer,kl_art_id_clean
73661,1,1.290.000,1.0,1294403,7,ST,KE,0,0,3180,NaT,20:06:13,20:11:37,1,981,10362,1294403
240004,1,1.290.000,1.0,1299925,2,ST,SC,0,1,2713,2024-10-08,09:19:11,09:22:00,1,684,22731,1299925
438510,1,1.290.000,1.0,1299925,3,ST,SC,0,1,2713,2024-10-08,09:19:11,09:22:00,1,684,22731,1299925
669166,1,1.290.000,1.0,1287614,1,ST,SC,0,0,4393,2024-07-08,12:42:17,12:43:59,1,303,2041,1287614
690113,1,1.290.000,1.0,20627328,1,ST,SC,1,1,6880,NaT,15:53:11,15:53:57,1,499,6681,20627328


In [281]:
### 6. Clean product cluster IDs (remove spaces, ensure strings) ###
sales['product_cluster_id_clean'] = sales['product_cluster_id'].astype(str).str.strip()
articles['product_cluster_id_clean'] = articles['product_cluster_id'].astype(str).str.strip()

In [282]:
# Check whether the product is in the sales
missing_articles_in_sales_1 = sales[~sales['product_cluster_id_clean'].isin(articles['product_cluster_id_clean'].unique())]

print(f"Number of rows with non-existent products: {len(missing_articles_in_sales_1)}")

Number of rows with non-existent products: 0


In [ ]:
# Check for consistent clarification products
articles[articles['product_cluster_id_clean'] == '1299925']

## Parse Transaction Dates

Convert `receipt_date` to datetime to enable time‑window filtering (e.g., last 90 days).

In [ ]:
### 7. Convert receipt_date to datetime format ###
sales['receipt_date'] = pd.to_datetime(sales['receipt_date'], errors='coerce')

## Recency Metric

Compute per‑customer last purchase date, then **Recency (days)** as:
`(global_max_last_purchase + 1 day) − last_purchase_date`.  
Higher score will correspond to **more recent** activity.

In [ ]:
### 8. Calculate Recency (days since last purchase per customer) ###

# Calculate last purchase date per customer
last_purchase = sales.groupby('customer_id')['receipt_date'].max()

# Define days since last order/purchase in the data set
last_purchase_to_max_date = last_purchase.max() + pd.Timedelta(days=1)

# Calculate recency – how many days have passed since the last purchase
recency = (last_purchase_to_max_date - last_purchase).dt.days

# data frame recency
recency_table = pd.DataFrame({'last_purchase': last_purchase, 'recency_day': recency})
recency_table

,last_purchase,recency_day
customer_id,,
1,2024-12-10,3.0
2,2024-11-02,41.0
3,2024-08-12,123.0
4,2024-11-10,33.0
5,2024-12-09,4.0
...,...,...
996,2024-12-08,5.0
997,2024-12-04,9.0
998,NaT,NaN


## Define 90‑Day Window & Frequency Metric

Set the analysis window to the last **90 days** relative to the max transaction date.  
Compute **Frequency** as the number of transactions per customer in this window.

In [ ]:
### 9. Calculate Frequency (number of transactions in the last 3 months) ###

max_date = sales['receipt_date'].max()

# Date 3 months earlier
three_months_ago = max_date - pd.Timedelta(days=90)

# Filter last purchases for 3 months
sales_last_3_months = sales[sales['receipt_date'] >= three_months_ago]

# Calculate Frequency
frequency = sales_last_3_months.groupby('customer_id')['transaction_id'].count().rename('frequency')

rec_freq = recency_table.join(frequency, how = 'left').fillna({'frequency':0}).astype({'frequency':int})

rec_freq.head()

,last_purchase,recency_day,frequency
customer_id,,,
1,2024-12-10,3.0,199
2,2024-11-02,41.0,37
3,2024-08-12,123.0,0
4,2024-11-10,33.0,22
5,2024-12-09,4.0,36


## Clean Sales Amounts (`sales_gross`)

Normalize numeric formats (e.g., values like `1.290.000` or `' 14.90 '`).  
This step ensures robust aggregation for **Monetary** and promo value metrics.

In [ ]:
# Check current type of sales data
sales_last_3_months['sales_gross'].dtype

sales_last_3_months['sales_gross'].head(10)

### 10. Clean sales_gross values (remove formatting inconsistencies) ###

def sales_gross_cleaning(value):
    if pd.isna(value):
        return pd.nan
    
    value = str(value).strip()

    # remove spaces
    value = value.replace(' ','')

    # If the value separated with two dos (eg: '1.290.000')
    if isinstance(value, str) and value.count('.') >= 2:
        first_dot = value.find('.')
        value = value[:first_dot] + value[first_dot+1:]  # удалить первую точку
        return value

    # If the value separated with one dot (например: '14.90', '0.6900')
    elif re.match(r'^\d+\.\d+$', value):
        try:
            return float(value)
        except:
            return np.nan

    # If just an integer
    elif value.isdigit():
        return float(value)

    return np.nan   # when not recognised

# leave only those lines where the number of points is not equal to 2
# sales = sales[sales['sales_gross'].astype(str).str.count('\.') != 2]

# Apply to column
sales['sales_gross_cleaned'] = sales['sales_gross'].apply(sales_gross_cleaning)

#test
print(sales[['sales_gross', 'sales_gross_cleaned']].head())

sales_last_3_months[['sales_gross', 'sales_gross_cleaned']].head()
#print(sales_last_3_months)
sales_last_3_months['sales_gross_cleaned'].dtype

  sales_gross sales_gross_cleaned
0      49.900                49.9
1      14.900                14.9
2      0.6900                0.69
3      21.800                21.8
4      13.900                13.9


dtype('O')

## Monetary Metric (90 Days)

Aggregate **sales_gross_cleaned** over the 90‑day window per customer to get **Monetary** (total spend).

In [ ]:
### 11. Calculate Monetary (total spend in the last 3 months) ###

# Force values to float before grouping
sales_last_3_months['sales_gross_cleaned'] = sales_last_3_months['sales_gross_cleaned'].astype(float)

# Grouping
monetary = sales_last_3_months.groupby('customer_id')['sales_gross_cleaned'].sum().round(2).rename('monetary')
monetary

/var/folders/4r/bw219wp92vz9b4fqkv_twq4m0000gn/T/ipykernel_13784/1369477423.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_last_3_months['sales_gross_cleaned'] = sales_last_3_months['sales_gross_cleaned'].astype(float)


,customer_id,Recency,Frequency,Monetary
0,1,3.0,199.0,1673.69
1,2,41.0,37.0,722.99
2,3,123.0,NaN,NaN
3,4,33.0,22.0,343.60
4,5,4.0,36.0,551.64
...,...,...,...,...
995,996,5.0,27.0,220.44
996,997,9.0,61.0,641.48
997,998,NaN,NaN,NaN
998,999,63.0,19.0,444.73


## Assemble RFM Table

Join **Recency**, **Frequency**, and **Monetary** into a single dataframe keyed by `customer_id`.

In [ ]:
# 12. Combine RFM metrics into a single dataframe
score_model = pd.DataFrame({'Recency': recency,'Frequency': frequency,'Monetary': monetary}).reset_index()

score_model

In [ ]:
#score_model = score_model.drop(columns=['promo_ratio'], errors='ignore')

## Promotion Engagement Metrics (90 Days)

From promo‑flagged transactions within the 90‑day window:
- **Promo Ratio** = promo transactions / total transactions  
- **Promo Value Ratio** = promo spend / total spend

These capture a customer’s propensity to react to promotions.

Merge Engagement Metrics into Score Model:

Left‑join the engagement ratios to the RFM table by `customer_id`.  
Fill missing ratios with 0 for customers without promo history in the window.

In [ ]:
### 13. Calculate promotion engagement ratios (count-based and value-based) ###

## Count-based calculation
# # Filter promo transactions
promo_last_3_months = promo.merge(sales_last_3_months, on=['customer_id', 'transaction_id'], how='inner')

# Count the number of all and promotional transactions per client
transaction_counts = sales_last_3_months.groupby('customer_id').size().rename('total_count_transactions')
promo_counts = promo_last_3_months.groupby('customer_id').size().rename('promo_count_transactions')

# Calculate the share
promo_ratio = (promo_counts/transaction_counts).fillna(0)

# Merge with RFM score model
score_model = score_model.merge(promo_ratio.rename('promo_ratio'), on='customer_id', how='left')

score_model

,customer_id,Recency,Frequency,Monetary,Recency_score,Frequency_score,Monetary_score,Promo_score,promo_value_ratio,Promo_Ratio_score,Promo_Value_score,Total_Score,promo_ratio
0,1,3.0,199.0,1673.69,5,5,4,2,0.37,3,3,20,0.623116
1,2,41.0,37.0,722.99,1,2,2,4,4.00,5,5,15,4.000000
2,4,33.0,22.0,343.60,2,1,1,4,3.44,5,5,14,3.454545
3,5,4.0,36.0,551.64,4,2,2,1,0.00,1,1,10,0.000000
4,6,40.0,145.0,2707.68,1,4,5,3,1.16,4,4,18,1.117241
...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,994,8.0,115.0,2671.61,3,4,5,2,0.00,2,2,16,0.000000
666,995,4.0,99.0,1155.01,4,4,3,3,1.93,4,4,19,1.666667
667,996,5.0,27.0,220.44,4,1,1,2,0.00,2,2,10,0.000000
668,997,9.0,61.0,641.48,3,3,2,4,4.86,5,5,18,4.754098


In [ ]:
#score_model = score_model.drop(columns=['promo_value_ratio'], errors='ignore')

In [269]:
## Value-based calculation
# 
# Filtering promotional transactions for the last 3 months
promo_last_3_months = promo.merge(sales_last_3_months, on=['customer_id', 'transaction_id'], how='inner')

# Promo share by amount (monetary value)
sales_last_3_months['sales_gross'] = pd.to_numeric(sales_last_3_months['sales_gross'], errors='coerce')
promo_last_3_months['sales_gross'] = pd.to_numeric(promo_last_3_months['sales_gross'], errors='coerce')

total_sales = sales_last_3_months.groupby('customer_id')['sales_gross'].sum().abs().rename('total_sales')
promo_sales = promo_last_3_months.groupby('customer_id')['sales_gross'].sum().abs().rename('promo_sales')
promo_value_ratio = (promo_sales / total_sales).fillna(0)

# Merge
score_model = score_model.merge(promo_value_ratio.rename('promo_value_ratio'), on='customer_id', how='left')
score_model

/var/folders/4r/bw219wp92vz9b4fqkv_twq4m0000gn/T/ipykernel_13784/3995970823.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_last_3_months['sales_gross'] = pd.to_numeric(sales_last_3_months['sales_gross'], errors='coerce')


,customer_id,Recency,Frequency,Monetary,Recency_score,Frequency_score,Monetary_score,Promo_score,Promo_Ratio_score,Promo_Value_score,Total_Score,promo_ratio,promo_value_ratio
0,1,3.0,199.0,1673.69,5,5,4,2,3,3,20,0.623116,0.366191
1,2,41.0,37.0,722.99,1,2,2,4,5,5,15,4.000000,4.000000
2,4,33.0,22.0,343.60,2,1,1,4,5,5,14,3.454545,3.435930
3,5,4.0,36.0,551.64,4,2,2,1,1,1,10,0.000000,0.000000
4,6,40.0,145.0,2707.68,1,4,5,3,4,4,18,1.117241,1.164062
...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,994,8.0,115.0,2671.61,3,4,5,2,2,2,16,0.000000,0.000000
666,995,4.0,99.0,1155.01,4,4,3,3,4,4,19,1.666667,1.933559
667,996,5.0,27.0,220.44,4,1,1,2,2,2,10,0.000000,0.000000
668,997,9.0,61.0,641.48,3,3,2,4,5,5,18,4.754098,4.864418


## Quintile Scoring (1–5 per Metric)

Assign scores:
- **Recency_score**: invert logic so lower days ⇒ higher score (labels `[5,4,3,2,1]` on raw Recency)
- **Frequency_score**, **Monetary_score**, **Promo_Ratio_score**, **Promo_Value_score**: higher is better (rank then `qcut`)
Handle duplicate edges with `duplicates='drop'`, then cast scores to integers.

In [ ]:
### 14. Assign quintile scores for each metric (5 = best, 1 = worst) ###
#
# Divide  customers into 5 tiers within each of the dimensions
# Ensure bins do not fail on duplicate edges
score_model['Recency_score'] = pd.qcut(score_model['Recency'], q=5, labels=[5, 4, 3, 2, 1], duplicates='drop')
score_model['Frequency_score'] = pd.qcut(score_model['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
score_model['Monetary_score'] = pd.qcut(score_model['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
score_model['Promo_Ratio_score'] = pd.qcut(score_model['promo_ratio'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')
score_model['Promo_Value_score'] = pd.qcut(score_model['promo_value_ratio'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5], duplicates='drop')

### 15. Calculate Total Score (sum of all metric scores) ###

# Drop NaNs caused by qcut inability to bin some values
score_model = score_model.dropna()

# Convert to int
score_model[['Recency_score', 'Frequency_score', 'Monetary_score', 'Promo_Ratio_score', 'Promo_Value_score']] = (score_model[['Recency_score', 'Frequency_score', 'Monetary_score', 'Promo_Ratio_score', 'Promo_Value_score']].astype(int))


score_model.head(10)

,customer_id,Recency,Frequency,Monetary,Recency_score,Frequency_score,Monetary_score,Promo_score,Promo_Ratio_score,Promo_Value_score,Total_Score,promo_ratio,promo_value_ratio
0,1,3.0,199.0,1673.69,5,5,4,2,3,3,20,0.623116,0.366191
1,2,41.0,37.0,722.99,1,2,2,4,5,5,15,4.000000,4.000000
2,4,33.0,22.0,343.60,2,1,1,4,5,5,14,3.454545,3.435930
3,5,4.0,36.0,551.64,4,2,2,1,1,1,10,0.000000,0.000000
4,6,40.0,145.0,2707.68,1,4,5,3,4,4,18,1.117241,1.164062
5,7,38.0,104.0,1462.95,1,4,3,1,1,1,10,0.000000,0.000000
6,8,6.0,151.0,2160.98,4,4,4,3,3,3,18,1.006623,0.893560
7,9,6.0,54.0,711.38,4,3,2,1,1,1,11,0.000000,0.000000
8,10,71.0,64.0,1335.79,1,3,3,1,1,1,9,0.000000,0.000000
9,12,38.0,106.0,2086.02,1,4,4,4,5,5,19,24.981132,20.845974


In [ ]:
#score_model = score_model.drop(columns=['promo_value_ratio','promo_ratio'], errors='ignore')

## Total Score & Ranking

Compute **Total_Score** as the sum of all five metric scores.  
Sort descending and extract the **Top‑100** customers as the target audience for the limited promotion.

In [274]:
#score_model = score_model.drop(columns=['promo_count_x', 'promo_count_y','promo_ratio_x','promo_count_transactions_x','has_promo_transaction','promo_count','promo_count_transactions_y','promo_ratio_y','Total_Score','Promo_Ratio_score'], errors='ignore')
#score_model

In [ ]:
### 16. Total Score ###
score_model['Total_Score'] = score_model[['Recency_score','Frequency_score','Monetary_score','Promo_Ratio_score','Promo_Value_score']].sum(axis=1)

# Select top 100 customers based on Total Score 
top_100 = score_model.sort_values('Total_Score', ascending=False).head(100)

top_100.head(10)

,customer_id,Recency,Frequency,Monetary,Recency_score,Frequency_score,Monetary_score,Promo_score,Promo_Ratio_score,Promo_Value_score,Total_Score,promo_ratio,promo_value_ratio
374,561,1.0,389.0,5977.93,5,5,5,4,5,5,25,7.730077,7.978073
621,921,3.0,176.0,2849.01,5,5,5,4,5,5,25,5.107955,4.757675
378,571,1.0,238.0,3895.03,5,5,5,4,5,5,25,6.247899,6.864795
382,580,3.0,318.0,5771.18,5,5,5,4,5,5,25,4.899371,3.784547
573,857,3.0,435.0,6916.59,5,5,5,4,5,5,25,12.963218,13.930711
138,200,1.0,188.0,2963.12,5,5,5,4,5,5,25,14.255319,9.605825
468,709,3.0,309.0,7519.96,5,5,5,4,5,5,25,22.414239,23.831711
216,332,3.0,322.0,5116.67,5,5,5,4,5,5,25,23.397516,15.183728
196,301,1.0,250.0,3647.28,5,5,5,4,5,5,25,3.844000,3.354326
122,170,1.0,199.0,3838.97,5,5,5,4,5,5,25,26.633166,24.692697


## Export Results

Write `top_100_customers.csv` with:
- `customer_id`
- RFM metrics and scores
- Promo engagement metrics and scores
- `Total_Score`

This file is ready for CRM activation and A/B testing.

In [ ]:
#top_100 = top_100.drop(columns=['Recency','Frequency','Monetary','promo_value_ratio','promo_ratio'], errors='ignore')
top_100 = top_100.merge(customer, on='customer_id', how = 'left')

top_100.head()

,customer_id,Recency,Frequency,Monetary,Recency_score,Frequency_score,Monetary_score,Promo_score,Promo_Ratio_score,Promo_Value_score,Total_Score,promo_ratio,promo_value_ratio,client_id,birth_year,household_size,preferred_store_id,registration_time
0,561,1.0,389.0,5977.93,5,5,5,4,5,5,25,7.730077,7.978073,1,1969,5.0,4853,2022-10-29
1,921,3.0,176.0,2849.01,5,5,5,4,5,5,25,5.107955,4.757675,1,1983,NaN,1523,2022-10-28
2,571,1.0,238.0,3895.03,5,5,5,4,5,5,25,6.247899,6.864795,1,1968,3.0,4490,2022-10-04
3,580,3.0,318.0,5771.18,5,5,5,4,5,5,25,4.899371,3.784547,1,0,2.0,3320,2022-11-13
4,857,3.0,435.0,6916.59,5,5,5,4,5,5,25,12.963218,13.930711,1,1987,5.0,3530,2022-11-11


In [277]:
#top_100.to_excel('top_100_customers.xlsx', index=False)
top_100.to_csv('top_100_customers.csv', index=False)

## Reproducibility & Notes

- The 90‑day window is configurable; adjust if promotion requires a different horizon.
- Consider tenant/market filtering via `mandant_id` if multi‑country data is combined.
- For stability, consider **winsorizing** extreme spend values before scoring.
- Next iteration: add **category‑level affinity** if promo targets specific product groups.